In [1]:
0

0

In [1]:
from fastai.vision.all import load_learner

In [2]:
model = load_learner('model.pkl')

c:\Users\shami\Tech\projects\quick-draw-proto\backend\model-api\.venv\Lib\site-packages\fastai\learner.py:455: UserWarning: load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you trust.
If you only need to load model weights and optimizer state, use the safe `Learner.load` instead.
  warn("load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you trust.\nIf you only need to load model weights and optimizer state, use the safe `Learner.load` instead.")


UnboundLocalError: cannot access local variable 'res' where it is not associated with a value

In [4]:
pred = model.predict("../data/testing/wheel/_78.png")
pred[0]

'wheel'

In [5]:
model.model.eval()
model.model.cpu()

Sequential(
  (0): TimmBody(
    (model): ConvNeXt(
      (stem): Sequential(
        (0): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
        (1): LayerNorm2d((96,), eps=1e-06, elementwise_affine=True)
      )
      (stages): Sequential(
        (0): ConvNeXtStage(
          (downsample): Identity()
          (blocks): Sequential(
            (0): ConvNeXtBlock(
              (conv_dw): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
              (norm): LayerNorm((96,), eps=1e-06, elementwise_affine=True)
              (mlp): Mlp(
                (fc1): Linear(in_features=96, out_features=384, bias=True)
                (act): GELU()
                (drop1): Dropout(p=0.0, inplace=False)
                (norm): Identity()
                (fc2): Linear(in_features=384, out_features=96, bias=True)
                (drop2): Dropout(p=0.0, inplace=False)
              )
              (shortcut): Identity()
              (drop_path): Identity()
          

In [6]:
# Check the device of the model's parameters
device = next(model.model.parameters()).device
print(device)  # Should print: cpu

cpu


In [7]:
import torch
import onnx

dummy_input = torch.randn(1, 1, 192, 192)

# Export to ONNX
onnx_path = "model_final.onnx"
torch.onnx.export(
    model.model,
    dummy_input,
    onnx_path,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
    opset_version=12
)

# Verify ONNX model
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)
print("ONNX model exported successfully!")

RuntimeError: Given groups=1, weight of size [96, 3, 4, 4], expected input[1, 1, 192, 192] to have 3 channels, but got 1 channels instead

In [26]:
import json


# Save vocabulary
vocab = list(model.dls.vocab)
with open("vocab.json", "w") as f:
    json.dump(vocab, f)
print("Vocabulary saved to vocab.json")

Vocabulary saved to vocab.json


In [9]:
# Check the model's architecture
print(model)  # Prints the entire model architecture

# Alternatively, check just the first layer (usually the first Conv2d layer)
print(model.model[0])  # This should give you information about the first layer

TimmBody(
  (model): ConvNeXt(
    (stem): Sequential(
      (0): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
      (1): LayerNorm2d((96,), eps=1e-06, elementwise_affine=True)
    )
    (stages): Sequential(
      (0): ConvNeXtStage(
        (downsample): Identity()
        (blocks): Sequential(
          (0): ConvNeXtBlock(
            (conv_dw): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
            (norm): LayerNorm((96,), eps=1e-06, elementwise_affine=True)
            (mlp): Mlp(
              (fc1): Linear(in_features=96, out_features=384, bias=True)
              (act): GELU()
              (drop1): Dropout(p=0.0, inplace=False)
              (norm): Identity()
              (fc2): Linear(in_features=384, out_features=96, bias=True)
              (drop2): Dropout(p=0.0, inplace=False)
            )
            (shortcut): Identity()
            (drop_path): Identity()
          )
          (1): ConvNeXtBlock(
            (conv_dw): Conv2d(9